In [1]:
! pip install -q langgraph

In [ ]:
import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END

class Brief(TypedDict):
    hours:     int                              # 수집 time window(시간)
    collected: list                             # ① 수집한 기사
    picked:    list                             # ② 선별해 남긴 다섯 건
    drafted:   Annotated[list, operator.add]    # ③ 취재한 초안 — 워커들이 나눠 채운다
    log:       Annotated[list, operator.add]    # 무슨 일이 있었는지

def collect(s: dict) -> dict:                  # ① 자료 수집  — 섹션 4에서 채운다
    return {"collected": [], "log": [f"① 수집   {s['hours']}시간 time window · 0건 (빈 노드)"]}

def select(s: dict) -> dict:                   # ② 중요도 선별 — 섹션 5에서 채운다
    return {"picked": [], "log": [f"② 선별   {len(s['collected'])} → 0건 (빈 노드)"]}

def report(s: dict) -> dict:                   # ③ 요약       — 섹션 6에서 채운다
    return {"drafted": [], "log": [f"③ 취재   {len(s['picked'])}건 (빈 노드)"]}

def verify(s: dict) -> dict:                   # ④ 검수       — 섹션 7에서 채운다
    return {"log": [f"④ 검수   {len(s['drafted'])}건 (빈 노드)"]}

def publish(s: dict) -> dict:                  # ⑤ 발행       — 섹션 8에서 채운다
    return {"log": [f"⑤ 발행   {len(s['drafted'])}건 (빈 노드)"]}

def build():                                    # 노드를 이름으로 찾아 붙인다
    g = StateGraph(Brief)
    for name in ("collect", "select", "report", "verify", "publish"):
        g.add_node(name, globals()[name])
    g.add_edge(START, "collect")
    g.add_edge("collect", "select")
    g.add_edge("select", "report")
    g.add_edge("report", "verify")
    g.add_edge("verify", "publish")
    g.add_edge("publish", END)
    return g

INIT = {"hours": 336,
        "collected": [], "picked": [], "drafted": [], "log": []}

def run():
    return build().compile().invoke(INIT)

for line in run()["log"]:
    print(line)

In [2]:
! pip install -q feedparser requests

In [ ]:
import feedparser, requests, re
from datetime import datetime, timezone

UA = {"User-Agent": "Mozilla/5.0 (newsletter-agent-course)"}
SOURCES = [
    ("I Care If You Listen", "https://icareifyoulisten.com/feed/"),
    ("Sequenza21",           "https://sequenza21.com/feed/"),
    ("VAN Magazine",         "https://van-magazine.com/feed/"),
    ("Guardian Classical",   "https://www.theguardian.com/music/classical-music-and-opera/rss"),
    ("Slipped Disc",         "https://slippedisc.com/feed/"),
]
now = datetime.now(timezone.utc)
print(f"{'소스':<22}{'상태':<6}{'건수':>5}{'요약길이':>9}{'최신글':>9}")
print("-" * 53)
for name, url in SOURCES:
    r = requests.get(url, headers=UA, timeout=20)
    f = feedparser.parse(r.content)
    e = f.entries[0]
    body = e.get("content", [{}])[0].get("value") if e.get("content") else e.get("summary", "")
    text = re.sub(r"<[^>]+>", "", body or "").strip()
    t = e.get("published_parsed")
    age = (now - datetime(*t[:6], tzinfo=timezone.utc)).total_seconds() / 3600 if t else None
    print(f"{name:<22}{r.status_code:<6}{len(f.entries):>5}{len(text):>9}"
          f"{(f'{age:.0f}h' if age is not None else '?'):>9}")

In [ ]:
import feedparser, requests, re
from datetime import datetime, timedelta, timezone

UA = {"User-Agent": "Mozilla/5.0 (newsletter-agent-course)"}
SOURCES = [
    ("I Care If You Listen", "https://icareifyoulisten.com/feed/"),
    ("Sequenza21",           "https://sequenza21.com/feed/"),
    ("VAN Magazine",         "https://van-magazine.com/feed/"),
    ("Guardian Classical",   "https://www.theguardian.com/music/classical-music-and-opera/rss"),
    ("Slipped Disc",         "https://slippedisc.com/feed/"),
]

# 일반 클래식 뉴스라 현대음악 글만 남겨야 하는 소스: 제목·요약에 키워드가 하나라도 있어야 통과
TOPIC_FILTER = {
    "Slipped Disc": re.compile(
        r"contemporary|new music|new work|world premiere|premiere[sd]?\b|composer|commission|"
        r"avant-garde|experimental|minimalis|electroacoustic|ensemble|festival|"
        r"\b(?:20th|21st)[- ]century|living composer",
        re.I),
}

def strip_tags(s):
    return re.sub(r"<[^>]+>", "", s or "").strip()

def published_at(entry):
    t = entry.get("published_parsed")
    return datetime(*t[:6], tzinfo=timezone.utc) if t else None

def collect(s: dict) -> dict:                   # ① 자료 수집 — 빈 노드를 갈아 끼운다
    cutoff = datetime.now(timezone.utc) - timedelta(hours=s["hours"])
    items, dead, empty, seen = [], [], [], set()
    filtered = 0
    for name, url in SOURCES:
        try:
            r = requests.get(url, headers=UA, timeout=20)
            r.raise_for_status()                # 403·404·500도 실패로 잡는다
            feed = feedparser.parse(r.content)
        except Exception:
            dead.append(name)                   # 한 곳이 죽어도 나머지는 계속
            continue
        n_before = len(items)
        for e in feed.entries:
            at = published_at(e)
            if not at or at < cutoff:           # time window 밖이거나 날짜가 없으면 버린다
                continue
            key = e.link.split("?")[0].rstrip("/")       # 추적용 꼬리표를 떼고 비교
            if key in seen:
                continue
            summary = strip_tags(e.get("summary", ""))[:300]
            topic = TOPIC_FILTER.get(name)
            if topic and not topic.search(f"{e.title} {summary}"):
                filtered += 1                   # 주제 밖이라 버린 글
                continue
            seen.add(key)
            items.append({"title": e.title, "url": e.link, "source": name, "at": at,
                          "summary": summary})
        if len(items) == n_before:
            empty.append(name)                  # 응답은 왔지만 window 안에 0건
    return {"collected": items,
            "log": [f"① 수집   {s['hours']}시간 time window · {len(items)}건"
                    + (f" · 주제 필터로 {filtered}건 제외" if filtered else "")
                    + (f" · 응답 없음 {dead}" if dead else "")
                    + (f" · 0건 {empty}" if empty else "")]}

for line in run()["log"]:                       # 그래프는 그대로, 노드만 바뀌었다
    print(line)

In [ ]:
! pip install -q openai feedparser requests
! pip install -q -U "brotli>=1.2.0"

In [ ]:
import os

# 방법 A) .env 파일 사용 (권장)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# 방법 B) Colab 보안 비밀 사용
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# 방법 C) 직접 입력 (노트북 공유 전 반드시 삭제)
# os.environ["OPENAI_API_KEY"] = "sk-..."

os.environ.setdefault("OPENAI_API_KEY", "")
print("키 설정 여부:", "설정됨" if os.environ.get("OPENAI_API_KEY") else "비어 있음 — 위 방법 중 하나로 채워주세요")

키 설정 여부: 설정됨


In [ ]:
import os, feedparser, requests
from collections import Counter
from openai import OpenAI

client = OpenAI(default_headers={"Accept-Encoding": "gzip, deflate"})  # 키는 위 셀에서 설정했습니다
UA = {"User-Agent": "Mozilla/5.0 (newsletter-agent-course)"}

# 4번 셀의 SOURCES를 그대로 쓴다 — 소스를 두 곳에서 따로 관리하면 어긋나기 때문
titles = []
for name, url in SOURCES:
    r = requests.get(url, headers=UA, timeout=20)
    r.raise_for_status()
    for e in feedparser.parse(r.content).entries[:3]:
        titles.append((name, e.title))

RUBRIC = ("당신은 현대음악(Contemporary Classical Music)을 챙겨 보는 독자를 위한 뉴스레터 편집자입니다.\n"
          "아래 기사 제목이 '이번 주 현대음악 흐름을 알려 줄 만한가'라는 "
          "기준에서 얼마나 중요한지 1~10점으로 매기세요. 숫자만 답하세요.")

scores = []
for src, t in titles:                      # ← 기사마다 한 번씩, 서로 모르는 채로
    r = client.chat.completions.create(
        model="gpt-4.1-mini", temperature=0, max_tokens=5,
        messages=[{"role": "system", "content": RUBRIC},
                  {"role": "user", "content": t}])
    scores.append((r.choices[0].message.content.strip(), src, t))

for s, src, t in sorted(scores, key=lambda x: -int(x[0])):
    print(f"{s:<6}{src:<22}{t[:44]}")
dist = Counter(s for s, _, _ in scores)
print("\n점수 분포:", " · ".join(f"{k}점 {v}건" for k, v in sorted(dist.items(), key=lambda kv: -int(kv[0]))))
print(f"서로 다른 점수 값: {len(dist)}개 / 후보 {len(scores)}건")


In [ ]:
# 17장 — 15장에서 따로 점수 매긴 같은 15건을, 이번엔 한 화면에 놓고 한 번에 비교한다
listing = "\n".join(f"{i}. [{s}] {t}" for i, (s, t) in enumerate(titles))
SYS = ("당신은 현대음악(Contemporary Classical Music)을 챙겨 보는 독자를 위한 뉴스레터 편집자입니다.\n"
       "아래 후보를 서로 비교해 '이번 주 현대음악 흐름을 알려 줄 만한' 순서대로 "
       "상위 5건을 고르세요. 번호만 쉼표로 구분해 답하세요. 설명 금지.")

r = client.chat.completions.create(                # ← 호출 한 번, 전부 한 화면에
    model="gpt-4.1-mini", temperature=0, max_tokens=40,
    messages=[{"role": "system", "content": SYS},
              {"role": "user", "content": listing}])
picked = [int(x) for x in r.choices[0].message.content.strip().replace(" ", "").split(",")][:5]

print("한 화면에 놓고 비교한 결과 — 상위 5건")
for rank, i in enumerate(picked, 1):
    print(f"  {rank}위  [{titles[i][0]}] {titles[i][1][:46]}")


In [ ]:
# 19장 — 구조화된 출력: 답의 모양(번호·이유·사건 라벨)을 스키마로 강제한다
from pydantic import BaseModel, Field

class Pick(BaseModel):
    index: int = Field(description="후보 목록에서의 번호")
    title: str = Field(description="고른 후보의 제목을 목록에 적힌 그대로 복사")
    reason: str = Field(description="왜 골랐는지 한국어 한 문장")
    event: str = Field(description="이 기사가 다루는 사건을 짧은 한국어 라벨로. 같은 사건이면 같은 라벨")

class Shortlist(BaseModel):
    picks: list[Pick]

SYS = ("당신은 현대음악(Contemporary Classical Music)을 챙겨 보는 독자를 위한 뉴스레터 편집자입니다.\n"
       "후보를 서로 비교해 '이번 주 현대음악 흐름을 알려 줄 만한' 기사 5건을 고르세요.\n"
       "각 기사에 대해 고른 이유와, 그 기사가 다루는 '사건' 라벨을 함께 답하세요.\n"
       "같은 사건을 다룬 기사에는 반드시 같은 라벨을 붙이세요.")

out = client.chat.completions.parse(            # ← parse: 스키마로 형식을 강제한다
    model="gpt-4.1-mini", temperature=0,
    messages=[{"role": "system", "content": SYS},
              {"role": "user", "content": listing}],
    response_format=Shortlist).choices[0].message.parsed

for p in out.picks:
    print(f"[{p.event:<22}] {titles[p.index][1][:40]}")
    print(f"{'':25}└ {p.reason[:60]}")


In [ ]:
# 20~21장 — select 노드: 묶음마다 예선, 살아남은 것을 한 화면에 놓고 본선 5건, 탈락자는 파일에 남긴다
import json
from collections import Counter
from datetime import datetime

BATCH, TARGET, PASS = 40, 5, 12    # 예선 묶음 크기, 최종 발행 건수, 묶음마다 통과 건수(매체 상한 때문에 넉넉히)
SPARE = 5                          # 같은 사건을 건너뛰고 채울 후보로, 본선에서 5건보다 더 받아 둔다
PER_SOURCE = 2                     # 한 매체에서 최종 발행할 수 있는 최대 건수

CRITERIA = ("독자는 현대음악(Contemporary Classical Music)을 챙겨 듣고 연주·작곡·기획하는 사람입니다.\n"
            "- 새 작품의 초연·위촉·신보 발매, 생존 작곡가·앙상블의 활동인가\n"
            "- 현대음악 페스티벌·기획, 분야의 흐름이나 논쟁(창작 환경·지원·기술)을 다루는가\n"
            "버릴 것: 바흐~말러 같은 정전 레퍼토리 중심 공연 리뷰, 현대음악과 무관한 부고·인사·스캔들, "
            "시즌·티켓 홍보성 소식")

def ask_picks(items, n):
    listing = "\n".join(f"{i}. [{it['source']}] {it['title']}" for i, it in enumerate(items))
    sys = (f"{CRITERIA}\n\n아래 목록에서 중요한 순서대로 {n}건을 고르세요.\n"
           "reason은 한국어 한 문장으로 쓰세요.\n"
           "같은 사건을 다룬 기사에는 같은 event 라벨을 붙이세요.")
    out = client.chat.completions.parse(
        model="gpt-4.1-mini", temperature=0,
        messages=[{"role": "system", "content": sys},
                  {"role": "user", "content": listing}],
        response_format=Shortlist).choices[0].message.parsed
    by_title = {it["title"]: i for i, it in enumerate(items)}
    seen, picks = set(), []
    for p in out.picks:                          # 없는 번호·겹친 번호는 버린다
        p.index = by_title.get(p.title.strip(), p.index)     # 제목이 일치하면 제목을 믿는다(번호는 가끔 어긋난다)
        if 0 <= p.index < len(items) and p.index not in seen:
            seen.add(p.index)
            picks.append(p)
    return picks

def select(s: dict) -> dict:                    # ② 중요도 선별 — 빈 노드를 갈아 끼운다
    items = s["collected"]
    if not items:
        return {"picked": [], "log": ["② 선별   0 → 0건"]}
    survivors, gates = [], []
    for i in range(0, len(items), BATCH):        # 예선 — 묶음마다 여덟 건
        chunk = items[i:i + BATCH]
        ranked = ask_picks(chunk, PASS + 1)      # 9등까지 받아 통과선(8등)과 첫 탈락(9등)을 남긴다
        survivors += [chunk[p.index] for p in ranked[:PASS]]
        gates.append({"pass_last": chunk[ranked[PASS - 1].index]["title"] if len(ranked) >= PASS else None,
                      "first_out": chunk[ranked[PASS].index]["title"] if len(ranked) > PASS else None})
    ranked = ask_picks(survivors, TARGET + SPARE)  # 본선 — 한 화면에 놓고 다섯 건 + 예비
    seen_events, per_src, finals, dropped = set(), Counter(), [], []
    for p in ranked:
        label = p.event.strip().lower()
        src = survivors[p.index]["source"]
        if len(finals) >= TARGET:
            dropped.append((p, "순위 밖"))
        elif label in seen_events:                  # 같은 사건은 한 건만
            dropped.append((p, "같은 사건"))
        elif per_src[src] >= PER_SOURCE:            # 한 매체가 자리를 다 차지하지 못하게
            dropped.append((p, "매체 상한"))
        else:
            seen_events.add(label)
            per_src[src] += 1
            finals.append(p)
    picked = [{**survivors[p.index], "event": p.event, "reason": p.reason} for p in finals]
    ratio = len(finals) / len(survivors)
    rec = {"at": datetime.now().isoformat(timespec="seconds"), "hours": s["hours"],
           "collected": len(items), "survivors": len(survivors), "ratio": round(ratio, 2),
           "gates": gates,
           "picked": [{"title": x["title"], "event": x["event"], "reason": x["reason"]} for x in picked],
           "dropped": [{"title": survivors[p.index]["title"], "event": p.event, "reason": p.reason,
                        "why": why} for p, why in dropped]}
    with open("select_log.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    return {"picked": picked,
            "log": [f"② 선별   {len(items)} → 예선 {len(survivors)} → {len(finals)}건 · 발행 비율 {ratio:.2f}"
                    + (f" · 5건 미달({len(finals)}건)" if len(finals) < TARGET else "")]}

result = run()
for line in result["log"]:
    print(line)
print()
for it in result["picked"]:
    print(f"[{it['event']:<20}] [{it['source']}] {it['title'][:50]}")
    print(f"{'':23}└ {it['reason']}")


In [ ]:
! pip install -q trafilatura


In [ ]:
# 22~24장 — 취재(report)의 요약은 headline · summary · why 세 칸으로 받는다
import re, trafilatura
from pydantic import BaseModel, Field

class Draft(BaseModel):
    headline: str = Field(description="20자 내외의 한국어 헤드라인")
    summary:  str = Field(description="세 문장 요약. ~합니다체, 과장 없이 건조하게. 작품명·인명은 원문 표기 그대로")
    why:      str = Field(description="현대음악을 챙겨 보는 독자에게 왜 중요한지 한 문장")

SYS_REPORT = ("당신은 현대음악(Contemporary Classical Music) 뉴스레터 기자입니다.\n"
              "아래 기사 본문을 읽고 헤드라인·요약·왜 중요한지를 한국어로 쓰세요.\n"
              "본문에 없는 사실·숫자·평가는 쓰지 마세요.\n"
              "'주목된다·기대를 모은다' 같은 기자체 표현은 쓰지 마세요.")

for it in result["picked"][:3]:                      # 10번 셀의 선별 결과
    downloaded = trafilatura.fetch_url(it["url"])
    body = trafilatura.extract(downloaded) if downloaded else None
    if not body or len(body) < 600:                  # ← 11장에서 정한 G1 기준선
        continue
    out = client.chat.completions.parse(
        model="gpt-4.1-mini", temperature=0,
        messages=[{"role": "system", "content": SYS_REPORT},
                  {"role": "user", "content": body[:6000]}],
        response_format=Draft).choices[0].message.parsed
    print("─" * 66)
    print("원문:", it["title"][:60], f"(본문 {len(body)}자)")
    print("  헤드라인:", out.headline)
    print("  요약    :", out.summary[:120], "…")
    print("  한글 포함:", bool(re.search(r"[가-힣]", out.headline + out.summary)))   # 24장 확인 예시


In [ ]:
# 25장 — report 노드: 기사 수만큼 Send로 펼친다 (fan-out / fan-in)
from langgraph.types import Send

class ReportIn(TypedDict):                           # 워커가 받는 것은 기사 하나뿐
    item: dict

def extract_body(url):
    d = trafilatura.fetch_url(url)
    return trafilatura.extract(d) if d else None

def draft(body):
    for _ in range(2):                               # 한국어로 안 나오면 한 번만 다시 부른다(24장)
        d = client.chat.completions.parse(
            model="gpt-4.1-mini", temperature=0,
            messages=[{"role": "system", "content": SYS_REPORT},
                      {"role": "user", "content": body[:6000]}],
            response_format=Draft).choices[0].message.parsed
        if re.search(r"[가-힣]", d.headline + d.summary):
            return d
    return None

def fan_report(s: dict):                             # 기사 수만큼 워커를 펼친다
    return [Send("report", {"item": it}) for it in s["picked"]] or "verify"   # 0건인 날에도 verify로 간다

def report(s: ReportIn) -> dict:                     # ③ 요약 — 기사 한 건을 맡는다
    it = s["item"]
    body = extract_body(it["url"])
    if not body or len(body) < 600:                  # ← 11장에서 정한 G1 기준선
        return {"drafted": [],
                "log": [f"   취재 제외 {it['source']} · 본문 {len(body or '')}자"]}
    d = draft(body)
    if d is None:
        return {"drafted": [], "log": [f"   취재 제외 {it['source']} · 한국어 요약 실패"]}
    return {"drafted": [{**it, "body": body[:6000], **d.model_dump()}]}

def build():                                         # 엣지 한 줄이 팬아웃으로 바뀐다
    g = StateGraph(Brief)
    for name in ("collect", "select", "report", "verify", "publish"):
        g.add_node(name, globals()[name])
    g.add_edge(START, "collect")
    g.add_edge("collect", "select")
    g.add_conditional_edges("select", fan_report, ["report", "verify"])   # ← 고정 엣지가 아니라
    g.add_edge("report", "verify")
    g.add_edge("verify", "publish")
    g.add_edge("publish", END)
    return g

result = run()
for line in result["log"]:
    print(line)
print()
for it in sorted(result["drafted"], key=lambda x: x["source"]):
    print(f"[{it['source']}] {it['headline']}")
    print(f"  요약: {it['summary']}")
    print(f"  왜  : {it['why']}\n")
